In [ ]:
from IPython.display import clear_output
%pip install --upgrade mech-interp-toolkit
clear_output()

In [ ]:
import os
os.chdir("..")

In [ ]:
from datasets import load_dataset
from typing import cast
import torch
import einops
from pathlib import Path

from utils.data import (
    extract_user_instruction,
)

from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed
from mech_interp_toolkit.activation_utils import get_embeddings_dict, get_activations, concat_activations

set_global_seed(0)
torch.enable_grad(False)

In [ ]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_train[:200]"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_path = "outputs/attacks/embedding_suffix/input/meta-llama-Llama-3.2-3B-Instruct/suffix.pt"
batch_size = 16

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load 200 examples from the circuit_breakers_train split
dataset = load_dataset(
    dataset_name,
    split=split,
)

dataset = cast(dict, dataset)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]

clear_output()

# Load model, tokenizer and config
model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",  # Scaled Dot Product Attention for efficiency
)

clear_output()


# load suffix
# from rashad's eval code
def load_suffix(suffix_path: str, device: torch.device):
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb


suffix_embed = load_suffix(suffix_path=suffix_path, device=device)
len_suffix = suffix_embed.shape[1]

In [ ]:
num_layers = config.num_hidden_layers
components = [(i, "layer_out") for i in range(num_layers)]

In [ ]:
# Iterate over prompts_str in batches
num_batches = (len(prompts_str) + batch_size - 1) // batch_size

base_collate = []
new_collate = []

for batch_idx in range(num_batches):
    start_idx = batch_idx * batch_size
    end_idx = min(start_idx + batch_size, len(prompts_str))

    batch_prompts = prompts_str[start_idx:end_idx]

    current_batch_size = len(batch_prompts)

    print(
        f"Processing batch {batch_idx + 1}/{num_batches} (samples {start_idx} to {end_idx - 1})"
    )

    batch_dict = ch_tokenizer(prompts=batch_prompts)
    # removes "input_ids" and adds "input_embeds"
    batch_embeds_dict = get_embeddings_dict(model, batch_dict)
    batch_embeds = batch_embeds_dict["input_embeds"]
    batch_attn_mask = batch_embeds_dict["attention_mask"]

    # broadcast suffix
    batch_suffix = einops.repeat(
        suffix_embed,
        "dummy pos d_model -> (curr_batch dummy) pos d_model",
        curr_batch=current_batch_size,
    )

    new_embeds = torch.cat(
        [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
    )
    attn_extension = torch.ones((current_batch_size, len_suffix)).to(device)
    new_attn = torch.cat([batch_attn_mask, attn_extension], dim=1)

    new_embeds_dict = {
        "input_embeds": new_embeds,
        "attention_mask": new_attn
    }

    base_acts = get_activations(
        model, inputs=batch_embeds_dict, layer_components=components, retain_grads=False, positions=-1
    ).cpu()
    
    new_acts = get_activations(
        model, inputs=new_embeds_dict, layer_components=components, retain_grads=False, positions=-1
    ).cpu()
    
    base_collate.append(base_acts)
    new_collate.append(new_acts)

In [ ]:
full_base_acts = dict(concat_activations(base_collate))
full_new_acts = dict(concat_activations(new_collate))

In [ ]:
save_path = Path("outputs/cached_activations/testing_200.pt")
save_path.parent.mkdir(parents=True, exist_ok=True)

# Save tuple of (full_base_acts, full_new_acts)
torch.save((full_base_acts, full_new_acts), save_path)
print(f"Saved activations to {save_path}")